# CryoEM Without Borders — single-particle pre-processing on FABRIC

The demo runs the first two steps of single-particle cryoEM pre-processing on a multi-GPU FABRIC node, using movies stored in the **cryoem-without-borders** project volume (the Globus collection).

1. **[MotionCor3](https://github.com/czimaginginstitute/MotionCor3)** aligns the dose-fractionated movie frames to remove beam-induced motion, using every GPU on the node.
2. **[CTFFIND4](https://grigoriefflab.umassmed.edu/ctf_estimation_ctffind_ctftilt)** estimates the contrast transfer function (defocus, astigmatism and fit resolution) of each corrected micrograph.
3. **Before vs. after:** for each micrograph, the raw frame sum is shown next to the motion-corrected micrograph, with both power spectra. Sharper Thon rings after correction show the quality gain.
4. *(Optional)* **[RELION 5.0](https://relion.readthedocs.io/en/release-5.0/)** runs the same steps through its own pipeline (Import → MotionCorr → CtfFind).
5. **Data transfer:** the processed results are sent from the GPU node to the downstream node at **FIU** (later replaced by **CAPE**). They travel over a dedicated FABRIC site-to-site link and land on the receiver's local disk, not the Ceph volume. The notebook reports throughput against an iperf3 baseline and verifies checksums at the far end.

```
  Ceph volume (input) ──► GPU node ──MotionCor3/CTFFIND4──► results ══ XFER L2 link ══► FIU receiver (→ CAPE)
```

**Prerequisite:** run the weave (`weave.sh`) first. It provisions the slice and writes `results/slice_info.json`. This notebook only connects to the existing slice. It never creates or deletes the slice unless you run the last cell yourself.

**Outputs.** On the GPU node, everything goes under `~/cryoem-work/<RUN_NAME>/`: `movies/` (symlinks), `mics/` (MotionCor3 micrographs, logs and `ctf/` fits), `mics_raw/` (unaligned sums). Figures and tables are downloaded into `results/<RUN_NAME>/` next to this notebook. The receiver gets a copy of `mics/` under `DEST_DIR`.

**Terms used below**

* **Movie:** one exposure recorded as many short frames (dose fractionation), so that sample motion during the exposure can be measured and undone.
* **EER:** Falcon's event-based movie format (Electron Event Representation). It stores hundreds of very thin frames, which are grouped into a few dozen fractions before alignment.
* **Gain reference:** the per-pixel sensitivity of the detector. Each frame is multiplied by it to remove fixed-pattern noise.
* **Dose weighting:** later frames are more radiation-damaged, so their high-resolution signal is down-weighted in the final sum.
* **CTF (contrast transfer function):** how the microscope's defocus and lens aberrations modulate the image. It must be measured for every micrograph before 3D reconstruction.
* **Power spectrum / Thon rings:** the 2D Fourier power of a micrograph. The CTF shows up as concentric rings; the further out they reach, the higher the resolution the image contains.

## 1. Configuration

All settings live in the next cell. For the UCSC ribosome data on the volume, the defaults work as they are.

* **Check for a new dataset:** `DATA_SUBDIR`, `MOVIE_EXT`, `GAIN_REF` and `MAX_MOVIES` (each EER movie is ~290 MB, and run time grows with the count).
* **Microscope values** (`PIXEL_SIZE`, `VOLTAGE_KV`, `TOTAL_DOSE`) are read from the EPU metadata when set to `"auto"`. Set them by hand only if §3 can't find the XML or reports wrong values.
* **Processing values** (`BFACTOR`, `PATCH`, `TRUNC`, the CTFFIND4 search range) come from the demo instructions. §5 and §6 explain what they do.
* **`RUN_NAME`** names the output folder. Use a new name to keep the results of an earlier run.
* **Transfer** settings (`DEST_SSH`, `DEST_DIR`, `INCLUDE_STACKS`, `XFER_METHOD`, `XFER_STREAMS`, `XFER_RUN`) are used only in §9. `XFER_RUN = "full"` sends the whole-dataset results from §7b instead of the demo batch.
* **Whole dataset:** §7b re-runs MotionCor3 + CTFFIND4 on every movie §3 finds (settings in that section).

In [1]:
# The slice is not hardcoded: §2 uses $SLICE_NAME, else results/slice_info.json from the
# weave, else the only slice in the project with a "cryoem" node. Override here if needed:
SLICE_NAME = None

# --- Data (paths are relative to the storage volume, mounted under /mnt/cephfs/...) ---
# Defaults match the UCSC ribosome session on the volume (EPU, Glacios, Falcon EER movies).
DATA_SUBDIR = "rawdata"     # sub-folder inside the volume that holds the movies ("" = whole volume)
MOVIE_EXT   = ".eer"        # .eer | .tif | .tiff | .mrcs | .mrc
MAX_MOVIES  = 8             # keep the demo short (~290 MB per EER movie)
GAIN_REF    = "auto"        # "auto" = the *.gain in the EPU session; "" = none; or a path inside the volume

# --- Microscope: "auto" reads the EPU XML that sits next to each movie ---
PIXEL_SIZE  = "auto"        # Å/pixel (this session: 0.713)
VOLTAGE_KV  = "auto"        # kV (Glacios: 200)
TOTAL_DOSE  = "auto"        # e/Å² per movie (this session: ~16.1)
CS_MM       = 2.7
AMP_CONTRAST = 0.1

# --- MotionCor3 (from the demo instructions) ---
BFACTOR     = 150           # -Bft
PATCH       = "1 1"         # -Patch: "1 1" = global only; "5 5" = local (anisotropic) correction
TRUNC       = 1             # -Trunc: drop last N fractions
OUT_STACK   = 1             # -OutStack 1 also writes the aligned stack (more data for the transfer)
EER_SAMPLING  = 1           # EER only: 1 = physical pixel, 2 = super-resolution
EER_FRACTIONS = 40          # EER only: group the raw EER frames into ~this many fractions

# --- CTFFIND4 search ---
CTF_BOX, RES_MIN, RES_MAX = 512, 50.0, 4.0
DF_MIN, DF_MAX, DF_STEP   = 5000.0, 35000.0, 500.0
ASTIG_RESTRAINT           = 100.0

RUN_NAME = "demo"                       # outputs go to ~/cryoem-work/<RUN_NAME>

# --- Transfer to the downstream site (FIU now, CAPE later) ---
DEST_SSH       = "receiver"             # ssh alias the weave set up on the GPU node (FIU VM over the XFER link).
                                        # For CAPE: an ssh alias/"user@host" reachable from the GPU node.
DEST_DIR       = "~/cryoem-inbox"       # receiver's LOCAL disk (not the Ceph volume)
INCLUDE_STACKS = True                   # also send the aligned movie stacks (much more data to move)
XFER_RUN       = None                   # None = RUN_NAME (demo batch); "full" = the whole dataset (§7b)
XFER_METHOD    = "rsyncd"               # "rsyncd" = rsync daemon on the receiver, plain TCP on the private XFER link
                                        # "ssh"    = rsync over SSH (needed on an untrusted path, e.g. CAPE over the internet)
XFER_STREAMS   = 16                     # parallel rsync streams (8 gave 9.57 Gbit/s MICH → FIU; 1 = one rsync)
RXDISK_GB      = 10                     # receiver disk write test before the transfer, GiB (0 = skip)
IPERF_SECONDS, IPERF_STREAMS = 10, 8    # network baseline before the transfer (after a 5 s ramp-up it ignores)

## 2. Connect to the slice

Finds the slice created by the weave and the GPU node in it, prints the node's SSH command (useful for watching a long step from a terminal), and lists its GPUs. MotionCor3 will use **every GPU listed**.

The map below shows the whole FABRIC footprint (every site and inter-site link; hover for bandwidth), the AmLight network that connects FABRIC at FIU to Latin America, Africa and Europe, and this slice on top: the GPU node, which reads the Ceph volume, and the receiver, joined by the dedicated XFER link used in §9. The purple star marks **CAPE** (Cape Town, South Africa), the downstream site that will replace the FIU receiver, with a dashed line for the planned transfer path. (FABRIC still lists CAPE at a placeholder address while it is in maintenance, so the map places it explicitly.)

In [ ]:
import json, os, time
from IPython.display import Image, display, Markdown
import fabric_env as F
import demo_viz as V
V.STAGES.clear()

fablib = F.get_fablib()
SLICE_NAME = SLICE_NAME or F.resolve_slice_name(fablib)
slice_obj = fablib.get_slice(name=SLICE_NAME)
node = slice_obj.get_node(name=F.NODE_NAME)
print(f"Slice {SLICE_NAME}: {slice_obj.get_state()}  lease ends {slice_obj.get_lease_end()}")
print(f"Node at {node.get_site()} ({node.get_host()})\nSSH: {node.get_ssh_command()}\n")

WORK = f"{F.REMOTE_WORK}/{RUN_NAME}"
LOCAL = os.path.join("results", RUN_NAME)
os.makedirs(LOCAL, exist_ok=True)

out, _ = node.execute("nvidia-smi --query-gpu=index,name,memory.total,utilization.gpu --format=csv", quiet=True)
print(out)
GPU_IDS = " ".join(l.split(",")[0].strip() for l in out.strip().splitlines()[1:])
print("MotionCor3 will use GPUs:", GPU_IDS)

In [ ]:
# Where everything runs: GPU node (reads the Ceph volume) and the downstream receiver.
fig_map = V.topology(fablib, slice_obj, ceph_cluster=F.CEPH_CLUSTER)
fig_map.show() if hasattr(fig_map, "to_html") else display(fig_map)

### FABRIC network

The FABRIC testbed: ESnet nodes on a terabit core, R&E facility nodes, and layer-2 links to Japan, Hawaii, Europe and South America. FIU/AMPATH is FABRIC's gateway to South America, and CAPE (Cape Town) is marked *coming soon*, reached from São Paulo:

![FABRIC network map](figures/fabric-map.png)


### AmLight

FABRIC's FIU site is AmLight's Miami hub, with AmLight facility ports (`AmLight-EXP-Layer2-FIU`, `AmLight-Layer3-FIU`). From there AmLight reaches Latin America, Africa (Sangano, Cape Town) and Europe (Sines):

![AmLight network map](figures/amlight-2.jpeg)

![AmLight network map, spectrum, leased and partner links](figures/amlight-1.png)


## 3. Browse the storage volume

The **cryoem-without-borders** CephFS volume is the same storage the Globus collection writes to. It is mounted under `/mnt/cephfs/…`, and anything uploaded through Globus shows up there. Movies are read in place.

In [ ]:
DATA_ROOT = F.data_dir(node)                       # e.g. /mnt/cephfs/west/<user>/<project>_cryoem-without-borders
DATA = f"{DATA_ROOT}/{DATA_SUBDIR}".rstrip("/")
print("Volume mounted at", DATA_ROOT)
node.execute(f"df -h {DATA_ROOT}/ | tail -1; echo; ls -lh {DATA}/ | head -40")
out, _ = node.execute(f"find -L {DATA}/ -maxdepth 8 -type f -name '*{MOVIE_EXT}' | sort", quiet=True)
all_movies = [m for m in out.strip().splitlines() if m]      # §7b processes all of them
print(f"\n{len(all_movies)} '{MOVIE_EXT}' movie(s) found; the demo batch uses the first "
      f"{min(MAX_MOVIES, len(all_movies))} (§7b runs the whole dataset).")
movies = all_movies[:MAX_MOVIES]
for m in movies:
    print("  ", m)

### Acquisition parameters

The next cell works out the values MotionCor3 and CTFFIND4 need, and prints them in a table. Check the table before going on: a wrong pixel size or dose silently gives wrong results.

* **Pixel size, voltage, total dose:** read from the `<movie>.xml` file that EPU writes next to every movie (the first movie stands in for the whole session).
* **Raw frames per movie:** counted from the first movie (TIFF pages for EER/TIFF, sections for MRC). For EER this is several hundred.
* **Gain reference:** with `GAIN_REF = "auto"`, the first `*.gain` file in the session folder.
* **Micrograph pixel size:** the same as the physical pixel unless `EER_SAMPLING = 2` (super-resolution), which halves it.

In [ ]:
# Acquisition parameters: EPU writes <movie>.xml next to every movie; "auto" values come from it.
import re
assert movies, "No movies found — check DATA_SUBDIR / MOVIE_EXT, or upload data through Globus."
m0 = movies[0]
stem0 = re.sub(r"(_EER)?(_fractions)?\.(eer|tiff?|mrcs?)$", "", m0)
meta = {}
try:
    node.download_file(os.path.join(LOCAL, "epu_meta.xml"), stem0 + ".xml")
    xml = open(os.path.join(LOCAL, "epu_meta.xml")).read()
    px = re.search(r"<pixelSize>.*?<numericValue>([^<]+)<", xml, re.S)
    kv = re.search(r"<AccelerationVoltage>([^<]+)<", xml)
    dose = re.search(r"TotalDose</a:Key>\s*<a:Value[^>]*>([^<]+)<", xml)
    meta = {"pixel_A": float(px.group(1)) * 1e10 if px else None,
            "kV": float(kv.group(1)) / 1000 if kv else None,
            "dose": float(dose.group(1)) if dose else None}
except Exception as exc:
    print("No EPU XML next to the movie:", exc)

PIXEL_SIZE = meta.get("pixel_A") if PIXEL_SIZE == "auto" else PIXEL_SIZE
VOLTAGE_KV = meta.get("kV") if VOLTAGE_KV == "auto" else VOLTAGE_KV
TOTAL_DOSE = meta.get("dose") if TOTAL_DOSE == "auto" else TOTAL_DOSE
assert PIXEL_SIZE and VOLTAGE_KV, "Set PIXEL_SIZE / VOLTAGE_KV in §1 (no EPU XML found)."

# Raw frames per movie (EER/TIFF pages, or MRC sections).
out, _ = node.execute(f'''{F.REMOTE_PY} -c "
import sys
p = '{m0}'
if p.endswith(('.mrc', '.mrcs')):
    import mrcfile; m = mrcfile.mmap(p, permissive=True); print(m.data.shape[0])
else:
    import tifffile; print(len(tifffile.TiffFile(p).pages))
"''', quiet=True)
N_FRAMES = int(out.strip().splitlines()[-1])

if GAIN_REF == "auto":
    out, _ = node.execute(f"find {DATA}/ -maxdepth 3 -name '*.gain' | head -1", quiet=True)
    GAIN = out.strip() or None
else:
    GAIN = f"{DATA_ROOT}/{GAIN_REF}" if GAIN_REF else None

OUT_PIX = PIXEL_SIZE / EER_SAMPLING if MOVIE_EXT == ".eer" else PIXEL_SIZE   # pixel size of the micrographs
display(Markdown(f"| parameter | value |\n|---|---|\n"
                 f"| pixel size | {PIXEL_SIZE:.4f} Å (micrographs {OUT_PIX:.4f} Å) |\n"
                 f"| voltage | {VOLTAGE_KV:g} kV |\n| total dose | {TOTAL_DOSE} e/Å² |\n"
                 f"| raw frames / movie | {N_FRAMES} |\n| gain reference | {GAIN or 'none'} |"))

## 4. Stage the selected movies

The movies stay on the volume. The work directory only holds symlinks to them, plus the outputs on local disk.

In [ ]:
links = " ".join(f"ln -sfn '{m}' {WORK}/movies/;" for m in movies)
_t = time.time()
node.execute(f"rm -rf {WORK} && mkdir -p {WORK}/movies {WORK}/mics/logs {WORK}/mics/ctf {WORK}/mics_raw && {links} ls -l {WORK}/movies", quiet=False)
out, _ = node.execute(f"du -shL {WORK}/movies | cut -f1", quiet=True)
V.STAGES["Stage movies from Ceph volume"] = (_t, time.time())
print("Input movies:", out.strip())

## 5. MotionCor3: frame alignment on all GPUs

MotionCor3 measures how the image drifts from frame to frame, shifts the frames back into register and sums them into one sharp micrograph. In serial mode (`-Serial 1`), it processes every movie in `movies/` and spreads them across the GPUs listed in `-Gpu`.

**Parameters**

* `-Patch 1 1`: global (whole-frame) alignment only. `5 5` also corrects local, uneven motion, but is slower.
* `-Bft 150`: B-factor (Å²) that low-pass filters the frames during alignment, so the drift is measured from low-resolution features that are well above the noise.
* `-Trunc 1`: drops the last fraction, which is the most radiation-damaged.
* `-OutStack 1`: also writes the aligned frame stack. It's large, and it's sent in §9 when `INCLUDE_STACKS = True`.
* **EER only:** `fmint.txt` holds one line, `<raw frames> <frames per fraction> <dose per raw frame>`. It groups the raw EER frames into about `EER_FRACTIONS` fractions and gives the dose used for dose weighting. `-EerSampling` picks physical (1) or super-resolution (2) pixels.

**Outputs in `mics/`** (per movie): `<movie>.mrc` (aligned sum), `<movie>_DW.mrc` (dose-weighted sum, when a dose is known), `<movie>_Stk.mrc` (aligned stack) and, in `mics/logs/`, the measured drift per fraction.

**What to expect**

* This is the longest step. Decoding EER is slow, so allow a minute or more per movie. Progress lines are printed as each movie finishes.
* For EER input, libtiff prints `TIFFReadDirectory: Warning, Unknown field with tag 65001/65002/65100`. These are Thermo Fisher's private EER tags, read by MotionCor3 itself, and are safe to ignore (they're filtered from the output).
* The cell then runs MotionCor3 a second time with `-Align 0` into `mics_raw/`. This gives the same gain-corrected frames summed **without** alignment, which is the "before" image in §7.

In [ ]:
in_flag = {".tif": "-InTiff", ".tiff": "-InTiff", ".eer": "-InEer"}.get(MOVIE_EXT, "-InMrc")
common = (f"{in_flag} {WORK}/movies/ -InSuffix {MOVIE_EXT} -Serial 1 "
          f"-PixSize {PIXEL_SIZE} -kV {VOLTAGE_KV:g} -Cs {CS_MM} -Trunc {TRUNC} -Gpu {GPU_IDS}")
if MOVIE_EXT == ".eer":
    # EER: group raw frames into fractions; the 3rd column is the dose per RAW frame.
    group = max(1, N_FRAMES // EER_FRACTIONS)
    per_frame = (TOTAL_DOSE or 0) / N_FRAMES
    node.execute(f"echo '{N_FRAMES} {group} {per_frame:.6f}' > {WORK}/fmint.txt", quiet=True)
    common += f" -FmIntFile {WORK}/fmint.txt -EerSampling {EER_SAMPLING}"
elif TOTAL_DOSE:
    common += f" -FmDose {TOTAL_DOSE / N_FRAMES:.4f}"
if GAIN:
    common += f" -Gain {GAIN}"

cmd = (f"MotionCor3 {common} -OutMrc {WORK}/mics/ -LogDir {WORK}/mics/logs/ "
       f"-Bft {BFACTOR} -OutStack {OUT_STACK} -Patch {PATCH}")
print(cmd, "\n")

t0 = time.time()
# stdbuf: MotionCor3 block-buffers stdout into a pipe, so without it only libtiff's
# (harmless) EER-tag warnings on stderr show up until the run ends.
node.execute(f"cd {WORK} && stdbuf -oL -eL {cmd} 2>&1 | grep --line-buffered -v 'TIFFReadDirectory: Warning' "
             f"| tee {WORK}/motioncor3.log")
mc_seconds = time.time() - t0
V.STAGES[f"MotionCor3 alignment ({len(GPU_IDS.split())} GPUs)"] = (t0, t0 + mc_seconds)
_t = time.time()

# "Before" reference: the same gain-corrected frames, summed WITHOUT alignment (-Align 0).
node.execute(f"cd {WORK} && MotionCor3 {common} -OutMrc {WORK}/mics_raw/ -Align 0 -OutStack 0 "
             f"> {WORK}/motioncor3_noalign.log 2>&1; ls {WORK}/mics_raw | wc -l", quiet=False)
V.STAGES["MotionCor3 unaligned sum (reference)"] = (_t, time.time())
node.execute(f"ls -lh {WORK}/mics/*.mrc")
print(f"\nMotionCor3: {len(movies)} movie(s) in {mc_seconds:.0f} s on GPU(s) {GPU_IDS}")

## 6. CTFFIND4: CTF estimation

CTFFIND4 fits a model of the contrast transfer function to the power spectrum of each micrograph. It reads its parameters from stdin (the here-doc from the demo instructions), runs once per micrograph, and runs in parallel across the CPU cores. It uses the plain aligned sums, not the `_DW` ones, because dose weighting removes the high-resolution Thon rings the fit depends on.

Each line of output gives:

* **Defocus U / V (Å):** defocus along the two axes of the astigmatism ellipse. They should fall inside the search range (`DF_MIN`–`DF_MAX`, 0.5–3.5 µm by default). A value at the edge of the range means the fit failed.
* **Astig angle (°):** the direction of the ellipse. The U–V difference matters more: a few hundred Å is normal.
* **Fit (Å):** the finest resolution to which the model still matches the Thon rings. Lower is better. Better than ~5 Å is good, and micrographs worse than ~8 Å are usually discarded.

Full fits (diagnostic image, `.txt` and log) are in `mics/ctf/`.

In [ ]:
W = WORK.replace("~", "$HOME")      # ~ is not expanded inside a here-doc
ctf_script = f'''#!/bin/bash
mic="$1"; stem=$(basename "$mic" .mrc)
ctffind > {W}/mics/ctf/$stem.log 2>&1 << EOF
$mic
{W}/mics/ctf/${{stem}}_ctf.mrc
{OUT_PIX}
{VOLTAGE_KV}
{CS_MM}
{AMP_CONTRAST}
{CTF_BOX}
{RES_MIN}
{RES_MAX}
{DF_MIN}
{DF_MAX}
{DF_STEP}
no
no
yes
{ASTIG_RESTRAINT}
no
no
EOF
tail -n 1 {W}/mics/ctf/${{stem}}_ctf.txt | awk -v s=$stem '{{printf "%-40s defocus %8.0f / %8.0f A  astig %6.1f deg  fit %5.2f A\\n", s, $2, $3, $4, $7}}'
'''
node.execute(f"cat > {WORK}/ctf_one.sh << 'XEOF'\n{ctf_script}\nXEOF\nchmod +x {WORK}/ctf_one.sh", quiet=True)

t0 = time.time()
# Plain aligned sums only: not the _DW/_DWS sums, the stacks, or MotionCor3's own _Ctf.mrc.
node.execute(f"ls {WORK}/mics/*.mrc | grep -Eiv '_(DW|DWS|Stk|Ctf)\\.mrc$' | xargs -P $(nproc) -n 1 {WORK}/ctf_one.sh")
V.STAGES["CTFFIND4 (all cores)"] = (t0, time.time())
print(f"\nCTFFIND4 finished in {time.time() - t0:.0f} s")

## 7. Before vs. after

* **Left:** the same gain-corrected frames summed without alignment (MotionCor3 `-Align 0`). Beam-induced motion blurs it, and its power spectrum fades at high resolution.
* **Middle:** the MotionCor3 micrograph. Thon rings reach further out in its power spectrum.
* **Right:** the CTFFIND4 fit (top) and the measured drift track of the frames (bottom).

For each movie, a second figure shows a full-resolution crop and the signal-vs-resolution curve. The shaded **10–5 Å** band is where motion blur hurts most. The "signal ×" value is the power after alignment divided by the power before, in that band. Values above 1 mean alignment recovered high-resolution signal, and values near 1 mean the movie barely moved.

In [ ]:
_t = time.time()
node.execute(f"{F.REMOTE_PY} ~/tools/cryoem_preview.py {WORK}/movies {WORK}/mics "
             f"--raw-dir {WORK}/mics_raw --pixel {OUT_PIX} --max {MAX_MOVIES}")
V.STAGES["Before/after previews"] = (_t, time.time())
# SFTP paths are relative to the home directory (no "~").
node.download_file(os.path.join(LOCAL, "summary.json"), f"{WORK[2:]}/mics/summary.json")
summary = json.load(open(os.path.join(LOCAL, "summary.json")))
for s in summary:
    png = os.path.join(LOCAL, os.path.basename(s["figure"]))
    node.download_file(png, s["figure"])          # absolute path on the node
    display(Image(filename=png))
    if s.get("zoom_figure"):                      # full-resolution crop + signal vs. resolution
        zpng = os.path.join(LOCAL, os.path.basename(s["zoom_figure"]))
        node.download_file(zpng, s["zoom_figure"])
        display(Image(filename=zpng))

In [ ]:
import pandas as pd
rows = []
for s in summary:
    c = s["ctf"] or {}
    rows.append({"micrograph": s["micrograph"],
                 "total drift (px)": round(s["total_drift_px"], 1) if s["total_drift_px"] else None,
                 "defocus U (µm)": round(c.get("defocus1_A", float("nan")) / 1e4, 3),
                 "defocus V (µm)": round(c.get("defocus2_A", float("nan")) / 1e4, 3),
                 "astig angle (°)": c.get("astig_angle_deg"),
                 "CTF fit res (Å)": c.get("fit_resolution_A"),
                 "signal gain 10–5 Å (×)": round(s["hf_gain"], 2) if s.get("hf_gain") else None})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(LOCAL, "ctf_summary.csv"), index=False)
display(Markdown(f"**{len(df)} micrographs** — MotionCor3 {mc_seconds:.0f} s on GPU(s) {GPU_IDS}"))
df

### Batch summary

One dashboard for the whole batch: motion removed per movie, the high-resolution signal recovered by alignment, CTF fit quality, and the average signal-vs-resolution curve before and after.

How to read the table above and the dashboard:

* **Total drift (px):** how far the image moved over the whole exposure, in micrograph pixels (multiply by the pixel size for Å). A few to a few tens of Å is typical. A movie that drifts far more than the rest of the batch is usually worth discarding.
* **Signal gain 10–5 Å (×):** as in §7. Larger drift should go with larger gain.
* **CTF fit res (Å):** as in §6, lower is better.

In [ ]:
fig_dash = V.dashboard(summary, mc_seconds, GPU_IDS, pixel_a=OUT_PIX)
fig_dash.savefig(os.path.join(LOCAL, "dashboard.png"), dpi=110, bbox_inches="tight")

## 7b. Scale up: the whole dataset

The demo batch above uses `MAX_MOVIES` movies. This section runs the same MotionCor3 and CTFFIND4 settings on **every movie §3 found**, then compares the time with the demo batch.

* **Runs on the node, detached.** The job keeps going if the notebook disconnects. The next cell starts it; the cell after reports progress and waits (interrupt it any time and run it again later).
* **In batches that resume:** MotionCor3 runs as a fresh process for every `FULL_BATCH` movies. One long process can fail part-way: the first full run stopped with `CUFFT_INTERNAL_ERROR` after 909 movies. A batch counts only if MotionCor3 exits cleanly and every movie has its aligned sum. A failed batch is retried once; if it fails again, the run stops and the progress cell shows why. Run the start cell again to continue: movies already aligned are skipped, and so are CTF fits already done (`FULL_RESTART = True` starts over).
* **Same settings, two differences:** no aligned stacks (`-OutStack 0`; at 2.6 GB per movie they would not fit on the disk) and no unaligned reference sums or preview figures. Writing stacks also costs time, so the demo's seconds per movie are a slight over-estimate for this run.
* **Disk:** MotionCor3 writes three ~65 MB micrographs per movie: the aligned sum (used by CTFFIND4), `_DW` (dose-weighted, what RELION uses) and `_DWS`. `_DWS` is deleted as soon as it is written (`FULL_KEEP_DWS = False`), so the run needs about 0.13 GB per movie: ~760 GB for the 5,658 movies on the volume. With the default 500 GB disks that does not fit: set `FULL_MAX` (about 3,000 movies fit) or run the weave with `DISK=1000` (FABRIC's maximum). The cell checks free space first and stops if it will not fit.
* Outputs go to `~/cryoem-work/<FULL_RUN_NAME>/`; the comparison is saved in `results/<FULL_RUN_NAME>/`.


In [ ]:
FULL_RUN_NAME = "full"     # outputs in ~/cryoem-work/<FULL_RUN_NAME>
FULL_MAX      = None       # None = every movie §3 found; or a number to cap the run
FULL_BATCH    = 50         # movies per MotionCor3 process (a failed batch is retried once, then the run stops)
FULL_RESTART  = False      # True = discard an earlier full run and start again; False = continue it
FULL_KEEP_DWS = False      # True = keep the _DWS micrographs too (~1.15 TB: more than a VM disk holds)

full_movies = all_movies[:FULL_MAX] if FULL_MAX else list(all_movies)
FULL_WORK = f"{F.REMOTE_WORK}/{FULL_RUN_NAME}"
FULL_REL = FULL_WORK.replace("~/", "")              # SFTP paths are relative to $HOME
FULL_LOCAL = os.path.join("results", FULL_RUN_NAME)
os.makedirs(FULL_LOCAL, exist_ok=True)

out, _ = node.execute(f"cd {FULL_WORK} 2>/dev/null && test -f run.pid && kill -0 $(cat run.pid) 2>/dev/null "
                      f"&& ! test -f DONE && ! test -f FAILED && echo RUNNING; cd {FULL_WORK} 2>/dev/null && {{ test -s done.txt && cat done.txt || "
                      # a run from before batching has no done.txt: its aligned sums count as done
                      f"ls mics/*.mrc 2>/dev/null | grep -Eiv '_(DW|DWS|Stk|Ctf)\\.mrc$' | "
                      f"sed 's#.*/##; s#\\.mrc$#{MOVIE_EXT}#'; }}", quiet=True)
running = "RUNNING" in out.split()
if running and not FULL_RESTART:
    print("A full run is already running. Go to the next cell to follow it "
          "(set FULL_RESTART = True to stop it and start over).")
else:
    done_before = set() if FULL_RESTART else set(out.split()) & {os.path.basename(m) for m in full_movies}
    todo = len(full_movies) - len(done_before)

    # Disk check for the movies still to align: per-movie output size measured on the demo batch
    # (without stacks, and without _DWS unless it is kept).
    skip_dws = "" if FULL_KEEP_DWS else "! -name '*_DWS.mrc'"
    out, _ = node.execute(f"find {WORK}/mics -maxdepth 1 -name '*.mrc' ! -name '*_Stk.mrc' {skip_dws} -printf '%s\\n' "
                          f"| awk '{{s+=$1}} END {{print s+0}}'; df -B1 --output=avail $HOME | tail -1", quiet=True)
    demo_out_bytes, free_bytes = (int(x) for x in out.split()[-2:])
    need_gb = demo_out_bytes / max(len(movies), 1) * todo / 1e9
    print(f"{len(full_movies)} movies, {len(done_before)} already aligned, {todo} to go; "
          f"expected new output ~{need_gb:.0f} GB, free {free_bytes / 1e9:.0f} GB")
    assert need_gb < 0.9 * free_bytes / 1e9, "Not enough disk: set FULL_MAX, or run the weave with a larger DISK."

    if FULL_RESTART:
        # Stop an earlier run: its script, then MotionCor3/CTFFIND (pattern [c]... so pkill
        # does not match its own command line), and remove its outputs.
        node.execute(f"test -f {FULL_WORK}/run.pid && kill $(cat {FULL_WORK}/run.pid) 2>/dev/null; true", quiet=True)
        node.execute(f"pkill -f '[c]ryoem-work/{FULL_RUN_NAME}/'; true", quiet=True)
        node.execute(f"rm -rf {FULL_WORK}", quiet=True)
    node.execute(f"mkdir -p {FULL_WORK}/movies {FULL_WORK}/mics/logs {FULL_WORK}/mics/ctf", quiet=True)

    # Movie list, CTFFIND helper and the run script go up as files (thousands of paths
    # do not fit on one command line).
    W_FULL = FULL_WORK.replace("~", "$HOME")
    with open(os.path.join(FULL_LOCAL, "movies.txt"), "w") as fh:
        fh.write("\n".join(full_movies) + "\n")
    with open(os.path.join(FULL_LOCAL, "ctf_one.sh"), "w") as fh:
        fh.write(ctf_script.replace(W, W_FULL))                 # §6's script, new paths
    common_full = common.replace(f"{WORK}/", f"{FULL_WORK}/")  # §5's flags, new paths
    mc_full_cmd = (f"MotionCor3 {common_full} -OutMrc {FULL_WORK}/mics/ -LogDir {FULL_WORK}/mics/logs/ "
                   f"-Bft {BFACTOR} -OutStack 0 -Patch {PATCH}").replace(FULL_WORK, W_FULL)
    # MotionCor3 reads a whole folder, so each batch gets its own folder of links (batch/). It
    # runs as a fresh process per batch: one long process can fail part-way (the first full
    # run hit CUFFT_INTERNAL_ERROR after 909 movies). A batch counts as done only if
    # MotionCor3 exits 0 AND every movie has its aligned sum; its movie names then go into
    # done.txt, which a later run skips. mc_batches.txt keeps "movies bytes seconds" per batch.
    mc_full_cmd = mc_full_cmd.replace(f"-InEer {W_FULL}/movies/", f"-InEer {W_FULL}/batch/") \
                             .replace(f"-InTiff {W_FULL}/movies/", f"-InTiff {W_FULL}/batch/") \
                             .replace(f"-InMrc {W_FULL}/movies/", f"-InMrc {W_FULL}/batch/")
    assert f"{W_FULL}/batch/" in mc_full_cmd, mc_full_cmd
    with open(os.path.join(FULL_LOCAL, "run.sh"), "w") as fh:
        fh.write(f"""#!/bin/bash
    cd {W_FULL}
    rm -f DONE FAILED t_mc_end t_ctf_end
    xargs -a movies.txt -d '\\n' ln -sfn -t movies/
    {'cp ' + W + '/fmint.txt ' + W_FULL + '/fmint.txt' if MOVIE_EXT == '.eer' else ':'}
    touch done.txt mc_batches.txt
    # A run from before batching has no done.txt entries: count movies with an aligned sum as done.
    if [ ! -s done.txt ]; then
        for m in mics/*.mrc; do s=$(basename "$m" .mrc)
            [[ $s =~ _(DW|DWS|Stk|Ctf)$ ]] || {{ [ -s "$m" ] && echo "$s{MOVIE_EXT}"; }}
        done >> done.txt
    fi
    test -f t_mc_start || date +%s.%N > t_mc_start
    {'' if FULL_KEEP_DWS else "( while [ ! -f t_mc_end ]; do find mics -maxdepth 1 -name '*_DWS.mrc' -mmin +1 -delete; sleep 30; done ) & WATCH=$!; trap 'kill $WATCH 2>/dev/null' EXIT"}
    fail() {{ echo "$1" > FAILED; date +%s.%N > t_mc_end; exit 1; }}
    while true; do
        sed 's#.*/##' movies.txt | grep -vxF -f done.txt | head -n {FULL_BATCH} > batch.txt
        [ -s batch.txt ] || break
        for try in 1 2; do
            rm -rf batch && mkdir batch && (cd batch && xargs -a ../batch.txt -d '\\n' -I{{}} ln -s ../movies/{{}} .)
            echo "=== batch of $(wc -l < batch.txt), try $try, $(date)" >> motioncor3.log
            t0=$(date +%s.%N)
            stdbuf -oL -eL {mc_full_cmd} 2>&1 | grep --line-buffered -v 'TIFFReadDirectory: Warning' >> motioncor3.log
            rc=${{PIPESTATUS[0]}}; t1=$(date +%s.%N)
            missing=$(sed 's/{MOVIE_EXT}$//' batch.txt | while read -r s; do [ -s "mics/$s.mrc" ] || echo "$s"; done | wc -l)
            [ "$rc" = 0 ] && [ "$missing" = 0 ] && break
            echo "=== batch failed: exit $rc, $missing movie(s) without output" >> motioncor3.log
            [ $try = 2 ] && fail "MotionCor3 failed twice on the same batch (exit $rc, $missing movie(s) without output); see motioncor3.log"
        done
        echo "$(wc -l < batch.txt) $(du -cbL batch | tail -1 | cut -f1) $(awk -v a=$t0 -v b=$t1 'BEGIN {{print b - a}}')" >> mc_batches.txt
        cat batch.txt >> done.txt
    done
    rm -rf batch batch.txt
    date +%s.%N > t_mc_end
    {'' if FULL_KEEP_DWS else "find mics -maxdepth 1 -name '*_DWS.mrc' -delete"}
    date +%s.%N > t_ctf_start
    # CTFFIND4 on every aligned sum that has no fit yet.
    ls mics/*.mrc | grep -Eiv '_(DW|DWS|Stk|Ctf)\\.mrc$' | while read -r m; do
        [ -s "mics/ctf/$(basename "$m" .mrc)_ctf.txt" ] || echo "$m"; done \\
      | xargs -r -P $(nproc) -n 1 ./ctf_one.sh >> ctffind.log 2>&1
    date +%s.%N > t_ctf_end
    touch DONE
    """)
    for f in ("movies.txt", "ctf_one.sh", "run.sh"):
        node.upload_file(os.path.join(FULL_LOCAL, f), f"{FULL_REL}/{f}")
    # Fully detached (no fd tied to this SSH session), so execute() returns at once.
    node.execute(f"cd {FULL_WORK} && chmod +x run.sh ctf_one.sh && "
                 f"{{ setsid nohup ./run.sh > run.out 2>&1 < /dev/null & echo $! > run.pid; }}", quiet=True)
    print(f"Started: {todo} movies in batches of {FULL_BATCH} on GPU(s) {GPU_IDS}. Log: {FULL_WORK}/motioncor3.log")
    print("Demo pace was", f"{mc_seconds / len(movies):.1f} s/movie, so expect roughly "
          f"{mc_seconds / len(movies) * todo / 60:.0f} min of MotionCor3.")


In [ ]:
# Progress. Waits until the run finishes or fails; interrupt any time — the job keeps running on the node.
WAIT_FOR_FULL = True
N_FULL = len(full_movies)
while True:
    out, _ = node.execute(
        f"cd {FULL_WORK} && echo $(sort -u done.txt 2>/dev/null | wc -l) "
        f"$(ls mics/ctf/*_ctf.txt 2>/dev/null | wc -l) "
        f"$(awk '{{n+=$1; s+=$3}} END {{print n+0, s+0}}' mc_batches.txt 2>/dev/null || echo 0 0) "
        f"$(test -f DONE && echo DONE || {{ test -f FAILED && echo FAILED; }} || "
        f"{{ test -f run.pid && kill -0 $(cat run.pid) 2>/dev/null && echo RUN; }} || echo STOPPED)", quiet=True)
    n_mc, n_ctf, n_timed, mc_s, state = out.split()[-5:]
    n_mc, n_ctf, n_timed, mc_s = int(n_mc), int(n_ctf), int(n_timed), float(mc_s)
    eta = (f", ~{mc_s / n_timed * (N_FULL - n_mc) / 60:.0f} min of alignment left"
           if n_timed and n_mc < N_FULL else "")
    print(f"{time.strftime('%H:%M:%S')}  aligned {n_mc}/{N_FULL}  CTF {n_ctf}/{N_FULL}  "
          f"MotionCor3 {mc_s / 60:.1f} min{eta}", flush=True)
    if state != "RUN" or not WAIT_FOR_FULL:
        break
    time.sleep(60)
if state == "DONE":
    print("Full run finished.")
elif state == "FAILED":
    out, _ = node.execute(f"cd {FULL_WORK} && cat FAILED && echo && tail -n 8 motioncor3.log", quiet=True)
    print("Full run STOPPED on an error:\n" + out +
          "\nFix the cause, then run the previous cell again: it continues where this run stopped.")
elif state == "STOPPED":
    print("The run is not running and did not finish. Run the previous cell again to continue it.")
else:
    print("Still running — run this cell again later.")


### Demo batch vs. whole dataset

Times are measured on the node (start/end stamps around each MotionCor3 batch and the CTFFIND4 step), so they do not include notebook overhead or pauses between a failed run and its continuation. *Predicted* is the demo's seconds per movie times the number of movies: a full run close to the prediction means the work scales linearly; a faster one means fixed start-up costs were a large share of the demo.


In [ ]:
import numpy as np
# MotionCor3 time and input size are summed over the batches (mc_batches.txt: movies, bytes,
# seconds), so pauses between a failed run and its continuation are not counted. Movies
# aligned before batching existed have no timing and are left out of the rates.
out, _ = node.execute(f"cd {FULL_WORK} && cat t_ctf_start t_ctf_end && "
                      f"awk '{{n+=$1; b+=$2; s+=$3}} END {{print n+0, b+0, s+0}}' mc_batches.txt && "
                      f"du -cbL {WORK}/movies | tail -1 | cut -f1 && "
                      f"for f in mics/ctf/*_ctf.txt; do tail -n 1 $f | awk '{{print $7}}'; done", quiet=True)
vals = out.split()
t_ctf0, t_ctf1 = float(vals[0]), float(vals[1])
n_full, full_in, mc_full = int(vals[2]), int(vals[3]), float(vals[4])
demo_in = int(vals[5])
fits = np.array([float(v) for v in vals[6:]])   # every aligned movie; CTFFIND4 reports 717.66 Å when it finds no fit
ctf_full = t_ctf1 - t_ctf0
ctf_demo = (lambda s: s[1] - s[0])(V.STAGES.get("CTFFIND4 (all cores)", (0, 0)))
demo_fits = np.array([s["ctf"]["fit_resolution_A"] for s in summary if s.get("ctf")])

def stats(n, in_bytes, mc_s, ctf_s, fit):
    return {"movies": n, "input GB": round(in_bytes / 1e9, 1),
            "MotionCor3 (s)": round(mc_s), "s per movie": round(mc_s / n, 2),
            "movies / min": round(60 * n / mc_s, 1), "input GB / min": round(60 * in_bytes / 1e9 / mc_s, 2),
            "CTFFIND4 (s)": round(ctf_s, 1), "median CTF fit (Å)": round(float(np.median(fit)), 2),
            "fits ≤ 5 Å": f"{(fit <= 5).sum()}/{len(fit)}"}

scaling = {"demo": stats(len(movies), demo_in, mc_seconds, ctf_demo, demo_fits),
           "whole dataset": stats(n_full, full_in, mc_full, ctf_full, fits)}
scaling["whole dataset"]["movies with a CTF fit"] = len(fits)
predicted = mc_seconds / len(movies) * n_full
scaling["whole dataset"]["predicted from demo (s)"] = round(predicted)
display(pd.DataFrame(scaling))
json.dump(scaling, open(os.path.join(FULL_LOCAL, "scaling.json"), "w"), indent=2, ensure_ascii=False)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].bar(["predicted\nfrom demo", "measured"], [predicted / 60, mc_full / 60], color=[V.GREY, V.BLUE])
ax[0].set_title(f"MotionCor3, {n_full} movies (min)")
ax[1].bar([f"demo\n({len(movies)})", f"whole dataset\n({n_full})"],
          [mc_seconds / len(movies), mc_full / n_full], color=[V.GREY, V.BLUE])
ax[1].set_title(f"Seconds per movie on {len(GPU_IDS.split())} GPU(s)")
ax[2].hist(np.clip(fits, None, 20), bins=32, range=(3, 20), color=V.GREEN)
ax[2].axvline(np.median(fits), color="k", ls="--", label=f"median {np.median(fits):.2f} Å")
ax[2].set_title(f"CTF fit resolution (Å); {(fits > 20).sum()} worse than 20 Å or no fit, at 20")
ax[2].legend(frameon=False)
for a in ax:
    V._clean(a)
fig.suptitle(f"Demo vs. whole dataset: {n_full} movies in {mc_full / 60:.1f} min "
             f"({60 * n_full / mc_full:.1f} movies/min)", weight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FULL_LOCAL, "scaling.png"), dpi=110, bbox_inches="tight")


## 8. (Optional) The same steps through RELION 5.0

RELION drives MotionCor3 and CTFFIND as external programs and writes standard STAR files, so later RELION jobs (picking, 2D classification) can start from here. Set `RUN_RELION = True` to run it. The weave must have been run with `INSTALL_RELION=true`.

The settings match §5. For EER, RELION does the frame grouping itself (`--eer_grouping`), so the dose it needs is per **grouped** fraction, not per raw frame. Outputs go to `~/cryoem-work/<RUN_NAME>/relion/`.

In [ ]:
RUN_RELION = False

if RUN_RELION:
    R = f"{WORK}/relion"
    gpus = ":".join(GPU_IDS.split())
    # Dose per movie fraction as RELION sees it. EER: RELION groups the raw frames itself,
    # so the dose is per group of raw frames (same grouping as §5's fmint.txt).
    eer_opts, frame_dose = "", (TOTAL_DOSE / N_FRAMES) if TOTAL_DOSE else None
    if MOVIE_EXT == ".eer":
        group = max(1, N_FRAMES // EER_FRACTIONS)
        eer_opts = f"--eer_grouping {group} --eer_upsampling {EER_SAMPLING}"
        frame_dose = frame_dose * group if frame_dose else None
    dose_opts = f"--dose_weighting --dose_per_frame {frame_dose:.4f}" if frame_dose else ""
    gain_opts = f"--gainref {GAIN}" if GAIN else ""
    node.execute(f'''set -e; export PATH=/opt/relion/bin:$PATH
mkdir -p {R}/Movies && cd {R} && ln -sfn {WORK}/movies/* Movies/
relion_import --do_movies --optics_group_name opticsGroup1 --angpix {PIXEL_SIZE} --kV {VOLTAGE_KV:g} \\
    --Cs {CS_MM} --Q0 {AMP_CONTRAST} --i "Movies/*{MOVIE_EXT}" --odir Import/job001/ --ofile movies.star
relion_run_motioncorr --i Import/job001/movies.star --o MotionCorr/job002/ --use_motioncor2 \\
    --motioncor2_exe /usr/local/bin/MotionCor3 --bin_factor 1 --bfactor {BFACTOR} \\
    --patch_x {PATCH.split()[0]} --patch_y {PATCH.split()[1]} --gpu "{gpus}" \\
    {eer_opts} {gain_opts} {dose_opts}
relion_run_ctffind --i MotionCorr/job002/corrected_micrographs.star --o CtfFind/job003/ \\
    --ctffind_exe /usr/local/bin/ctffind --is_ctffind4 --Box {CTF_BOX} --ResMin {RES_MIN} --ResMax {RES_MAX} \\
    --dFMin {DF_MIN} --dFMax {DF_MAX} --FStep {DF_STEP} --dAst {ASTIG_RESTRAINT}
head -40 CtfFind/job003/micrographs_ctf.star''')

## 9. Transfer the results to the downstream site (FIU → CAPE)

The processed data (micrographs, CTF fits, figures and, optionally, the aligned stacks) moves from the GPU node to the receiver over the dedicated **XFER** link. That link is a FABRIC site-to-site L2 circuit with its own private subnet, separate from the Ceph network the node reads its input from.

1. **Path:** the far end, RTT and MTU.
2. **Baseline:** an iperf3 memory-to-memory test over the same link, and a write test of the receiver's disk (a transfer can't be faster than the disk it lands on).
3. **Transfer:** a checksum manifest, then `XFER_STREAMS` parallel rsync streams, timed on the node.
4. **Verify:** `sha256sum -c` on the receiver.

**Why an rsync daemon (`XFER_METHOD = "rsyncd"`):** OpenSSH has a fixed ~2 MiB flow-control window per connection, so one rsync-over-SSH stream can't go faster than window / RTT, however well TCP is tuned (2 MiB / 30 ms ≈ 0.56 Gbit/s). The first run measured exactly that (0.54 of 11.72 Gbit/s with one stream), and 8 streams MICH → FIU reached 2.25 of 7.92 Gbit/s. The XFER link is a private L2 circuit, so the files can go over plain TCP to an rsync daemon on the receiver instead. Its socket buffers are then sized by the kernel's TCP autotuning (raised by the weave's network tuning). The daemon listens only on the receiver's XFER address, accepts only the GPU node's XFER address, and is stopped after the transfer. SSH is still used to control it. Set `XFER_METHOD = "ssh"` to compare the two, or for a path that needs encryption.

Set `XFER_RUN = "full"` in §1 to send the whole-dataset results from §7b (~760 GB) instead of the demo batch. Checksums are computed and verified on all cores, since one `sha256sum` over that much data takes a long time.

To switch to **CAPE** later, point `DEST_SSH` / `DEST_DIR` in §1 at CAPE and set `XFER_METHOD = "ssh"` (that path crosses the public internet, so it needs encryption; Globus is the faster option there). Nothing else in this section changes.

In [ ]:
# Resolve the dataplane addresses the ssh alias uses (HostName / BindAddress).
cfg_out, _ = node.execute(f"ssh -G {DEST_SSH}", quiet=True)
ssh_cfg = dict(l.split(" ", 1) for l in cfg_out.strip().splitlines() if " " in l)
dest_ip, src_ip = ssh_cfg.get("hostname"), ssh_cfg.get("bindaddress")
# Site, MTU and NIC come from the live slice (results/slice_info.json is written in the weave's run folder).
rx_node = next((n for n in slice_obj.get_nodes() if n.get_name() == DEST_SSH), None)
out, _ = node.execute(f"ip -o -4 addr show | awk -v ip={src_ip} 'index($4, ip\"/\")==1 {{print $2}}' "
                      f"| xargs -r -I{{}} cat /sys/class/net/{{}}/mtu", quiet=True)
try:
    xfer_nic = node.get_component(name="xfer_nic").get_model()
except Exception:
    xfer_nic = None
rx = {"site": rx_node.get_site() if rx_node else None,
      "mtu": int(out.split()[-1]) if out.split() else None, "nic": xfer_nic}
print(f"Source : {node.get_name()} @ {node.get_site()}  ({src_ip or rx.get('source_xfer_ip', '?')})")
print(f"Dest   : {DEST_SSH} @ {rx.get('site', '?')}  ({dest_ip})  -> {DEST_DIR}/{RUN_NAME}")
print(f"MTU    : {rx['mtu'] or '?'}   NIC: {rx['nic'] or '?'}")
node.execute(f"ping -c 5 -q {'-I ' + src_ip if src_ip else ''} {dest_ip} | tail -2; "
             f"ssh -o BatchMode=yes {DEST_SSH} 'hostname; df -h {DEST_DIR} | tail -1'")

# SSH (and therefore rsync) must run on the dataplane, not the management network.
out, _ = node.execute(f"ssh -o BatchMode=yes {DEST_SSH} 'echo SSHCONN $SSH_CONNECTION'", quiet=True)
conn = next(l.split()[1:] for l in out.splitlines() if l.startswith("SSHCONN "))
assert conn[2] == dest_ip and (not src_ip or conn[0] == src_ip), f"SSH is not on the dataplane: {conn}"
print(f"SSH session on dataplane: {conn[0]} -> {conn[2]}")

In [ ]:
# Network baseline (memory-to-memory) on the same path the files will take. -O 5 leaves the
# first 5 s out of the average: on this path TCP takes ~4 s to ramp up to full rate.
_t = time.time()
node.execute(f"ssh {DEST_SSH} 'pkill -x iperf3; iperf3 -s -D -1'", quiet=True)
time.sleep(1)
bind = f"-B {src_ip}" if src_ip else ""
out, _ = node.execute(f"iperf3 -c {dest_ip} {bind} -P {IPERF_STREAMS} -t {IPERF_SECONDS} -O 5 -J", quiet=True)
iperf = json.loads(out[out.index("{"):])
iperf_gbps = iperf["end"]["sum_received"]["bits_per_second"] / 1e9
V.STAGES["iperf3 baseline"] = (_t, time.time())
print(f"iperf3 baseline: {iperf_gbps:.2f} Gbit/s ({IPERF_STREAMS} streams, {IPERF_SECONDS} s), "
      f"retransmits {iperf['end']['sum_sent'].get('retransmits', 'n/a')}")

In [ ]:
# Receiver disk: how fast it can write, so a slow disk is not taken for a slow network.
# Random data (storage can skip zeros; 256 MiB of it, repeated), written with O_DIRECT so the page cache does not hide
# the disk; RXDISK_GB GiB are written to DEST_DIR, then deleted.
rxdisk_gbps = None
if RXDISK_GB:
    _t = time.time()
    out, _ = node.execute(f"ssh {DEST_SSH} 'mkdir -p {DEST_DIR} && cd {DEST_DIR} && "
                          f"head -c 256M /dev/urandom > /dev/shm/xfer-rand && t0=$(date +%s.%N) && "
                          f"for i in $(seq {4 * RXDISK_GB}); do cat /dev/shm/xfer-rand; done | "
                          f"dd of=.disktest bs=16M iflag=fullblock oflag=direct conv=fsync status=none && "
                          f"echo DISK $t0 $(date +%s.%N); rm -f .disktest /dev/shm/xfer-rand'", quiet=True)
    stamp = [l for l in out.splitlines() if l.startswith("DISK ")]
    assert stamp, f"Receiver disk test failed (set RXDISK_GB = 0 to skip it): {out.strip()[-300:]}"
    t0, t1 = (float(x) for x in stamp[-1].split()[1:])
    rxdisk_gbps = RXDISK_GB * 2**30 * 8 / (t1 - t0) / 1e9
    V.STAGES["Receiver disk test"] = (_t, time.time())
    print(f"Receiver disk write: {rxdisk_gbps:.2f} Gbit/s ({rxdisk_gbps / 8:.2f} GB/s, {RXDISK_GB} GiB, O_DIRECT)"
          + ("  <- slower than the network: the disk will limit large transfers" if rxdisk_gbps < iperf_gbps else ""))


In [ ]:
# Which run to send: the demo batch (RUN_NAME) unless XFER_RUN names another, e.g. "full" (§7b).
XRUN = XFER_RUN or RUN_NAME
SRC = f"{F.REMOTE_WORK}/{XRUN}/mics"
XLOCAL = os.path.join("results", XRUN)
os.makedirs(XLOCAL, exist_ok=True)
excl = "" if INCLUDE_STACKS else "! -name '*_Stk.mrc'"

# Checksum manifest + payload size. Hashing runs on every core, in batches of 16 files so each
# sha256sum writes its lines in one go and lines from different batches never interleave.
out, _ = node.execute(f"cd {SRC} && find . -type f ! -name MANIFEST.sha256 {excl} -print0 | "
                      f"xargs -0 -P $(nproc) -n 16 sha256sum > MANIFEST.sha256 && wc -l < MANIFEST.sha256 && "
                      f"find . -type f {excl} -printf '%s\\n' | awk '{{s+=$1}} END {{print s}}'", quiet=True)
n_files, n_bytes = (int(x) for x in out.split()[-2:])
print(f"Payload ({XRUN}): {n_files} files, {n_bytes / 1e9:.2f} GB (stacks {'included' if INCLUDE_STACKS else 'excluded'})")

# Transfer, timed on the node itself. The destination is emptied first and --whole-file
# is used, so every run moves the full payload (re-runs would otherwise skip files).
# The files are split into XFER_STREAMS lists (largest first, round-robin) and sent in parallel.
LISTS = f"/tmp/xfer-{XRUN}"
assert XFER_METHOD in ("rsyncd", "ssh"), XFER_METHOD
RSYNCD = "/tmp/rsyncd-xfer"                 # .conf / .pid / .log on the receiver
STOP_RSYNCD = f"test -f {RSYNCD}.pid && kill $(cat {RSYNCD}.pid); rm -f {RSYNCD}.pid {RSYNCD}.lock"
if XFER_METHOD == "rsyncd":
    # rsync daemon on the receiver: plain TCP on the private XFER link, so no SSH window limit.
    # It listens on the receiver's XFER address only and accepts only the GPU node's XFER address.
    RSYNCD_PORT = 8730
    conf = (f"use chroot = no\\nmax connections = {XFER_STREAMS + 4}\\npid file = {RSYNCD}.pid\\n"
            f"lock file = {RSYNCD}.lock\\n"   # default /var/run/rsyncd.lock needs root
            f"$U[xfer]\\n  path = $D\\n  read only = no\\n  hosts allow = {src_ip}\\n")
    node.execute(f"ssh {DEST_SSH} '{STOP_RSYNCD}; mkdir -p {DEST_DIR} && D=$(cd {DEST_DIR} && pwd)/{XRUN} && "
                 # only a root daemon switches users (to "nobody" by default); keep it as root then
                 f"U=$(test $(id -u) = 0 && echo \"uid = 0\\\\ngid = 0\\\\n\"); "
                 f"printf \"{conf}\" > {RSYNCD}.conf && rsync --daemon --config={RSYNCD}.conf "
                 f"--address={dest_ip} --port={RSYNCD_PORT} --log-file={RSYNCD}.log'", quiet=True)
    XFER_DEST = f"--address={src_ip} rsync://{dest_ip}:{RSYNCD_PORT}/xfer/"
else:
    # rsync over SSH: each stream is capped by SSH's ~2 MiB window / RTT, and one core for AES.
    RSH = "ssh -T -o Compression=no -c aes128-gcm@openssh.com"
    XFER_DEST = f"-e '{RSH}' {DEST_SSH}:{DEST_DIR}/{XRUN}/"
out, _ = node.execute(
    f"rm -rf {LISTS} && mkdir -p {LISTS} && cd {SRC} && find . -type f {excl} -printf '%s %P\\n' | sort -nr | "
    f"awk -v n={XFER_STREAMS} -v d={LISTS} '{{sub(/^[0-9]+ /, \"\"); print > (d \"/\" (NR-1)%n)}}' && "
    f"ssh {DEST_SSH} 'rm -rf {DEST_DIR}/{XRUN} && mkdir -p {DEST_DIR}/{XRUN}' && "
    f"t0=$(date +%s.%N) && ls {LISTS}/* | xargs -P {XFER_STREAMS} -I{{}} "
    f"rsync -a --whole-file --files-from={{}} {SRC}/ {XFER_DEST} && "
    f"t1=$(date +%s.%N) && awk -v a=$t0 -v b=$t1 'BEGIN {{print \"ELAPSED=\" b - a}}'")
if XFER_METHOD == "rsyncd":
    node.execute(f"ssh {DEST_SSH} '{STOP_RSYNCD}'", quiet=True)
elapsed = [l for l in out.splitlines() if l.startswith("ELAPSED=")]
assert elapsed, "Transfer failed (rsync errors above)" + (f"; receiver log: ssh {DEST_SSH} tail {RSYNCD}.log" if XFER_METHOD == "rsyncd" else "")
xfer_s = float(elapsed[-1].split("=")[1])
_t_end = time.time()
XLABEL = {"rsyncd": "rsync daemon", "ssh": "rsync/SSH"}[XFER_METHOD]
V.STAGES[f"Transfer to receiver ({XLABEL} x{XFER_STREAMS})"] = (_t_end - xfer_s, _t_end)
xfer_gbps = n_bytes * 8 / xfer_s / 1e9
print(f"\n{XLABEL} x{XFER_STREAMS}: {n_bytes / 1e9:.2f} GB in {xfer_s:.1f} s = {xfer_gbps:.2f} Gbit/s")


In [ ]:
# Verify at the far end.
_t = time.time()
# The manifest is split into one chunk per receiver core (no empty chunks) and checked in parallel.
out, _ = node.execute(
    f"ssh {DEST_SSH} 'cd {DEST_DIR}/{XRUN} && rm -rf /tmp/verify-{XRUN} && mkdir -p /tmp/verify-{XRUN} && "
    f"L=$(( ($(wc -l < MANIFEST.sha256) + $(nproc) - 1) / $(nproc) )) && "
    f"split -l $L MANIFEST.sha256 /tmp/verify-{XRUN}/m. && "
    f"ls /tmp/verify-{XRUN}/m.* | xargs -P $(nproc) -n 1 sha256sum -c --quiet && echo ALL_VERIFIED; du -sh .'",
    quiet=True)
verified = "ALL_VERIFIED" in out
V.STAGES["Checksum verify at receiver"] = (_t, time.time())
print(out)

transfer = {"run": XRUN, "source": f"{node.get_name()}@{node.get_site()}", "destination": f"{DEST_SSH}@{rx.get('site', dest_ip)}",
            "files": n_files, "GB": round(n_bytes / 1e9, 3), "seconds": round(xfer_s, 1),
            "rsync Gbit/s": round(xfer_gbps, 2), "iperf3 Gbit/s": round(iperf_gbps, 2),
            "efficiency": f"{100 * xfer_gbps / iperf_gbps:.0f}%" if iperf_gbps else None,
            "MTU": rx.get("mtu"), "NIC": rx.get("nic"), "method": XLABEL, "streams": XFER_STREAMS,
            "receiver disk Gbit/s": round(rxdisk_gbps, 2) if rxdisk_gbps else None, "checksums": "verified" if verified else "MISMATCH"}
json.dump(transfer, open(os.path.join(XLOCAL, "transfer.json"), "w"), indent=2)
display(pd.DataFrame([transfer]).T.rename(columns={0: "value"}))

In [ ]:
fig_x = V.transfer(iperf, transfer)
fig_x.savefig(os.path.join(XLOCAL, "transfer.png"), dpi=110, bbox_inches="tight")
fig_map = V.topology(fablib, slice_obj, transfer=transfer, ceph_cluster=F.CEPH_CLUSTER)   # map with measured rates
fig_map.show() if hasattr(fig_map, "to_html") else display(fig_map)

### End-to-end timeline

Every stage from reading the movies off the Ceph volume to a verified copy at the receiver: storage, GPU, CPU and network.

In [ ]:
fig_t = V.timeline()
fig_t.savefig(os.path.join(LOCAL, "timeline.png"), dpi=110, bbox_inches="tight")
print("Figures saved in", LOCAL)

## 10. Slice housekeeping

The slice is **kept** by default so the demo can be re-run without re-provisioning. Extend the lease before it runs out. Delete the slice only when you no longer need it.

In [ ]:
slice_obj.renew(days=14)            # FABRIC caps the lease at ~14 days from now
print("Lease ends:", slice_obj.get_lease_end())

In [ ]:
DELETE_SLICE = False                # set True and run to tear the slice down
if DELETE_SLICE:
    slice_obj.delete()
    print(f"Slice {SLICE_NAME} deleted.")